# AI-Agents and Agentic AI

## 1 Introduction to Smolagents

`smolagents` is one of the simplest and most lightweight frameworks for building *ReAct-style AI Agents*.  
It supports multiple Large Language Models (LLMs), including models from:

- Hugging Face Hub  
- OpenAI  
- Anthropic  
- And others

The library implements the **ReAct framework** and allows you to:

- Define custom tools
- Use pre-built tools
- Build multi-agent systems where agents collaborate on tasks

`smolagents` provides two main agent types:

- **ToolCallingAgent**: uses JSON/text-structured tool calls  
- **CodeAgent** — writes Python code as its actions and executes it in a sandbox

### 1.1 Installation

In [ ]:
%pip install 'smolagents[e2b]'
%pip install ddgs
%pip install yfinance

import os

os.environ["HF_TOKEN"] = "your_hf_token_here"


### 1.2 ToolCallingAgent

The `ToolCallingAgent` provides a direct implementation of the ReAct framework.
It generates:

1. A **reasoning step** (“Thought”)
2. An **action step**, expressed as a JSON tool call

It then selects and executes tools based on the reasoning step produced by the LLM.

### 1.3 CodeAgent

Unlike the `ToolCallingAgent`, which produces JSON tool calls,
the **CodeAgent writes Python code** as its action.
This code may:

* Call custom tools
* Perform arbitrary computations
* Use Python’s built-in capabilities

All the generated code is then **executed**, and anything printed (`print(...)`) is treated as an **observation** and fed back into the ReAct loop.

### 1.4 Example: ToolCallingAgent vs. CodeAgent Actions

**ToolCallingAgent Action (declares a tool call)**

```json
{
  "tool_call": {
    "name": "get_stock_price",
    "arguments": {
      "ticker": "AAPL"
    }
  }
}
```

**CodeAgent Action (executes code directly)**

```python
result = get_stock_price("AAPL")
print(result)
final_answer(result)
```

## 2 CodeAgent Example: Broker Agent

In [ ]:
import yfinance as yf
from smolagents import CodeAgent, InferenceClientModel, DuckDuckGoSearchTool, tool

# A simple wallet class to manage stocks and balance
class myWallet:
    def __init__(self, balance: float):
        self.balance = balance
        self.stocks = {}

    def add_stock(self, ticker: str, quantity: int, stock_price: float):
        if self.balance < quantity * stock_price:
            return {"status": "error", "message": "Insufficient balance to buy stocks."}
        
        self.balance -= quantity * stock_price
        if ticker in self.stocks:
            self.stocks[ticker] += quantity
        else:
            self.stocks[ticker] = quantity
        return {"status": "success", "balance": self.balance, "stocks": self.stocks}
    
    def remove_stock(self, ticker: str, quantity: int, stock_price: float):
        if ticker not in self.stocks or self.stocks[ticker] < quantity:
            return {"status": "error", "message": f"Not enough shares of {ticker} to sell."}
        
        self.balance += quantity * stock_price
        self.stocks[ticker] -= quantity
        if self.stocks[ticker] == 0:
            del self.stocks[ticker]
        return {"status": "success", "balance": self.balance, "stocks": self.stocks}
    


# Helper function to fetch stock price using yfinance
def get_stock_price(ticker: str) -> float:
    """
    Fetches the current stock price for the given ticker symbol.

    Args:
        ticker: Stock ticker symbol (e.g., "AAPL", "GOOGL").

    Returns:
        The current stock price as a float.
    """
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        price = info.get("regularMarketPrice")
        return price
    except Exception as e:
        print(f"Could not fetch stock price for {ticker}: {e}")
        return -1
    

wallet = myWallet(10000)  # Initialize wallet with $10,000

In [ ]:
@tool
def stock_price(ticker: str) -> str:
    """
    Fetches the current stock price for the given ticker symbol.

    Args:
        ticker: Stock ticker symbol (e.g., "AAPL", "GOOGL").

    Returns:
        The current stock price as a float, or an error message if the price cannot be fetched.
    """
    price = get_stock_price(ticker)
    if price > 0:
        return price
    else:
        return f"Could not fetch stock price for {ticker}."


@tool
def buy_stocks(ticker: str, quantity: int) -> str:
    """
    Buys a specified quantity of stocks for the given ticker symbol.
    
    Args:
        ticker: Stock ticker symbol (e.g., "AAPL", "GOOGL").
        quantity: Number of shares to buy.
    Returns:
        A string confirming the purchase or an error message.    
    """

    price = get_stock_price(ticker)
    
    if price > -1:
        response = wallet.add_stock(ticker, quantity, price)
        if response["status"] == "success":
            return f"Bought {quantity} shares of {ticker} at ${price:.2f} each. Remaining balance: ${response['balance']:.2f}"
        else:
            return response["message"]
    else:
        return f"Could not fetch stock price for {ticker}."

@tool
def get_portfolio() -> str:
    """
    Returns the current portfolio of stocks and balance.
    
    Returns:
        A string summarizing the current portfolio and balance.
    """
    stocks_summary = ", ".join([f"{ticker}: {quantity} shares" for ticker, quantity in wallet.stocks.items()])
    return f"Current Portfolio: {stocks_summary if stocks_summary else 'No stocks owned.'} | Balance: ${wallet.balance:.2f}"


model = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-7B-Instruct")

broker_agent = CodeAgent(model=model, tools=[stock_price, buy_stocks, get_portfolio, DuckDuckGoSearchTool(max_results=3)], executor_type="local", max_steps=5)

In [ ]:
broker_agent.run("What is the current price of AAPL stock?")
print("\n\nThe current price of AAPL stock is:", stock_price("AAPL"))

In [ ]:
broker_agent.run("Buy 10 shares of AAPL if the price is below $320.")
print("\n\nCurrent Portfolio:", get_portfolio())


#### Excercise 1:

Implement a new tool called `sell_stocks` that allows the agent to sell a specified quantity of stocks for a given ticker symbol. The tool should check if the agent owns enough shares to sell and update the wallet balance accordingly.

## 3 Code Execution: Local vs Sandbox

When a `CodeAgent` generates Python code, it needs an **executor** to run it.
`smolagents` supports two execution modes, selected via the `executor_type` parameter:

- **`local`** — the code runs directly inside the current Python process, in the same
  environment as the notebook. This is the simplest option and requires no additional
  setup, but it means the agent has unrestricted access to the local filesystem and
  installed packages, which can be risky for untrusted or destructive operations.
- **`sandbox`** — the code runs inside an isolated sandbox. The agent cannot
  touch local files, and the entire environment is discarded at the end of the session.
  This is the recommended option whenever the agent might write, modify, or delete files.

A possible sandbox provider is **E2B** ([e2b.dev](https://e2b.dev)), which provisions
a lightweight cloud micro-VM for each session. Files created inside the sandbox persist
across multiple `.run()` calls within the same session, but are permanently destroyed
when `.cleanup()` is called or the session ends.


### 3.1 Setting Up E2B

To use the E2B sandbox you need a free account and an API key.
The key is passed automatically to the executor via an environment variable,
so no additional configuration is required in the agent code itself.

1. Create a free account at [https://e2b.dev](https://e2b.dev)
2. Go to **Dashboard → API Keys** and generate a new key
3. Set `E2B_API_KEY` as an environment variable before creating the agent


In [ ]:
os.environ["E2B_API_KEY"] = "your_e2b_api_key_here"

### 3.2 Fixing a `smolagents` Compatibility Issue

The current version of `smolagents` has a known bug when using the E2B executor.
During agent initialisation, `RemotePythonExecutor` attempts to override the internal
`final_answer` tool a second time, even when it has already been patched, which raises
an `AttributeError` and prevents the agent from starting.

The function below applies a safe monkey-patch that checks whether the tool has already
been modified before trying to override it. Run this cell once before creating any
E2B-based agent.


In [ ]:
def fix_smolagents_patch():
    from smolagents.remote_executors import RemotePythonExecutor

    _original_patch_method = RemotePythonExecutor._patch_final_answer_with_exception

    def _safe_patch_final_answer(self, final_answer_tool):
        if hasattr(final_answer_tool, "_forward"):
            return
        _original_patch_method(self, final_answer_tool)

    RemotePythonExecutor._patch_final_answer_with_exception = _safe_patch_final_answer

fix_smolagents_patch()


### 3.3 PC Manager Agent

We can now build an agent that manages files inside the E2B sandbox.
The agent is equipped with four custom tools that map directly to common
filesystem operations:

- **`list_files`** — lists all files present in the sandbox working directory
- **`read_file`** — opens a file and returns its content as a string
- **`write_file`** — creates or appends to a file (`'w'` to overwrite, `'a'` to append)
- **`delete_file`** — removes a file from the sandbox

The agent also has access to `DuckDuckGoSearchTool`, so it can search the web
when a task requires up-to-date information alongside file operations.

Because `executor_type="e2b"` is set, all Python code generated by the agent
runs inside the remote sandbox — the local machine is never touched.


In [ ]:
from smolagents import CodeAgent, InferenceClientModel, tool, DuckDuckGoSearchTool


@tool
def list_files() -> str:
    """
    Lists all files in the system.
    Args:
        None
    Returns:
        A string listing all files in the current directory.
    """
    import os
    try:
        files = os.listdir(".")
        return "Files in the system:\n" + "\n".join(files)
    except Exception as e:
        return f"Error listing files: {e}"


@tool
def read_file(filename: str) -> str:
    """
    Reads the content of a specified file.
    Args:
        filename: The name of the file to read.
    Returns:
        The content of the file or an error message.
    """
    try:
        with open(filename, "r") as f:
            return f.read()
    except FileNotFoundError:
        return f"Error: File '{filename}' not found."
    except Exception as e:
        return f"Error reading file '{filename}': {e}"


@tool
def write_file(filename: str, content: str, open_type: str) -> str:
    """
    Writes content to a specified file.
    Args:
        filename:  The name of the file to write.
        content:   The content to write to the file.
        open_type: The mode to open the file ('w' for overwrite, 'a' for append).
    """
    if open_type not in ["w", "a"]:
        return "Error: open_type must be 'w' for overwrite or 'a' for append."
    try:
        with open(filename, open_type) as f:
            f.write(content)
        return f"File '{filename}' has been written successfully."
    except Exception as e:
        return f"Error writing file '{filename}': {e}"


@tool
def delete_file(filename: str) -> str:
    """
    Deletes a specified file from the sandbox.
    Args:
        filename: The name of the file to delete.
    """
    import os
    try:
        os.remove(filename)
        return f"File '{filename}' has been deleted successfully."
    except FileNotFoundError:
        return f"Error: File '{filename}' not found."
    except Exception as e:
        return f"Error deleting file '{filename}': {e}"


In [ ]:
model = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-7B-Instruct")

pc_manager_agent = CodeAgent(
    model=model,
    tools=[list_files, read_file, write_file, delete_file, DuckDuckGoSearchTool(max_results=3)],
    executor_type="e2b",
    executor_kwargs={"timeout": 1800}, # Set timeout to 30 minutes
    max_steps=10
)


In [ ]:
# Test 1 — write a file
content = "AI agents use LLMs to complete tasks autonomously."
pc_manager_agent.run(f"Write a file called notes.txt with the content: {content}.")

try:
    res = pc_manager_agent.python_executor.sandbox.files.read("notes.txt")
    if res == content:
        print("File content is correct.")
    else:
        print(f"notes.txt contains unexpected content: {res}")
except Exception as e:
    print(f"File not found: {e}")

In [ ]:
# Test 2 — read the file back
pc_manager_agent.run("Read the content of notes.txt")

In [ ]:
# Test 3 — delete and confirm
pc_manager_agent.run("Delete notes.txt and list all files to confirm it has been removed.")

try:
    res = pc_manager_agent.python_executor.sandbox.files.read("notes.txt")
    print(f"Unexpectedly found notes.txt with content: {res}")
except Exception as e:
    print(f"File not found as expected: {e}")


In [ ]:
# Cleanup the sandbox after testing
pc_manager_agent.cleanup()

#### Exercise 2:

The agent currently supports four tools: `list_files`, `read_file`, `write_file`,
and `delete_file`. Your task is to extend it with two additional tools and then
test the full system using an **interactive conversation loop**.

**Implement the following tools:**

- **`rename_file(old_name, new_name)`** — renames a file inside the sandbox using
  `os.rename`. It should return a clear error message if the source file does not exist.
- **`count_words(filename)`** — reads a file and returns the number of words it contains,
  for example: `"File 'notes.txt' contains 14 words."`. Handle the case where
  the file does not exist.

Once the tools are implemented, rebuild the agent with all six tools and run
the **interactive loop** in the next cell. The loop maintains a conversation history
so the agent remembers what happened in previous turns — you can refer to a file
created earlier without re-explaining the context.


In [ ]:
# TODO: implement rename_file and count_words, then run the loop in the next cell

@tool
def rename_file(old_name: str, new_name: str) -> str:
    """
    Renames a file in the sandbox.
    Args:
        old_name: The current name of the file.
        new_name: The new name for the file.
    Returns:
        A success or error message.
    """
    # TODO: use os.rename(old_name, new_name) and handle FileNotFoundError
    raise NotImplementedError("Implement rename_file!")


@tool
def count_words(filename: str) -> str:
    """
    Counts the number of words in a file.
    Args:
        filename: The name of the file to read.
    Returns:
        The word count as a string, or an error message.
    """
    # TODO: open the file, split its content on whitespace, return the count
    raise NotImplementedError("Implement count_words!")


# Rebuild the agent with all six tools once both functions are implemented
pc_manager_agent = CodeAgent(
    model=model,
    tools=[list_files, read_file, write_file, delete_file,
           #rename_file, count_words,           # TODO: your two new tools 
           DuckDuckGoSearchTool(max_results=3)],
    executor_type="e2b",
    executor_kwargs={"timeout": 1800}, # Set timeout to 30 minutes
    max_steps=10,
    verbosity_level=-1
)


In [ ]:
from collections import deque
# Interactive loop with conversation history
# The agent remembers previous turns — build on earlier steps without repeating context

MAX_HISTORY = 10

# Store conversation history
messages = deque(maxlen=MAX_HISTORY)


def build_prompt(messages, current_input):
    """
    Build a structured prompt including conversation history.
    """
    history = "\n".join(
        f"{msg['role'].capitalize()}: {msg['content']}"
        for msg in messages
    )

    return f"""
Conversation history:
{history}

Current user request:
{current_input}
""".strip()


def chat_loop(agent):
    print("Type 'exit' to quit.\n")

    while True:
        try:
            user_input = input("You: ").strip()

            if user_input.lower() == "exit":
                break

            if not user_input:
                print("Please enter a message.")
                continue
            print(f"User: {user_input}\n")
            # Save user message
            messages.append({
                "role": "user",
                "content": user_input
            })

            # Build prompt
            prompt = build_prompt(messages, user_input)

            # Run agent
            response = agent.run(prompt)

            # Save assistant response
            messages.append({
                "role": "assistant",
                "content": str(response)
            })

            print(f"Agent: {response}\n")

        except KeyboardInterrupt:
            print("\nExiting...")
            break

        except Exception as e:
            print(f"\nError: {e}\n")

    # Cleanup resources
    agent.cleanup()
    print("Cleanup completed.")


# Start the loop
chat_loop(pc_manager_agent)
